# news_company_reason — 예상 가격 방향에 맞는 기업 뉴스 1문장

아침 모델의 **종목별 예상 시가**(`model.card_stocks`의 `h`, 전일 종가 대비 비율)가 오르는 쪽(+)인지 내리는 쪽(−)인지에 맞춰, 그 기업이 나온 밤사이 뉴스 중 **영향(`company_impact`)이 같은 방향인 소식 하나**를 골라 OpenAI로 한 문장씩 정리한다.

```
model.card_stocks(서버 DB) ─ 06:20에 07:30 카드의 종목별 예상 시가 h
news_extract  ─ llm_source.jsonl에 기사 한 줄(entities, company_impact)
news_track    ─ 같은 줄에 story_id·news_id·source_key / importance.csv에 사건 매체 수
   ▼
news_company_reason.ipynb ─ 기업마다: h > 0이면 영향 positive, h < 0이면 negative인 소식 중 매체 수 1위
                             (없거나 h가 없으면 그 기업이 나온 모든 소식 중 매체 수 1위) → OpenAI 1문장(한 번 호출로 모든 기업)
   ▼  company_reasons.jsonl (기업마다 한 줄)
```

- 실행 시각: 오전 7:00(06:20 계산된 07:30 카드 뒤, 2026-11-20 정함). 서버 실행(`run_news.py`·cron)에 넣는 것은 팀원이 한다.
- 이 노트북은 `llm_source.jsonl`, `importance.csv`, DB의 `model.card_stocks`를 **읽기만** 한다.
- 프롬프트: `prompt/company_reason_prompt.md`(proj/prompt에서 복제).
- 기사를 고르는 방식(기업 이름 맞추기, 시간대 안 매체 수)은 `news_company_hot_issue.ipynb`와 같다.

## 1. 설정

- `START_KST`, `END_KST`, `TIME_COLUMN`: 볼 기사 범위. 기본은 브리핑과 같은 **전날 23:00 ~ 그날 05:45 수집분**(`first_seen_utc`, 2026-11-20 정함).
- `CARD_SLOT`: 예상 가격을 읽을 카드(`0730`: 06:20 계산). 날짜는 `END_KST`의 한국 날짜(`YYYYMMDD`)다.
- `COMPANIES`, `MENTIONS`, `MAX_ARTICLES_PER_ISSUE`: `news_company_hot_issue.ipynb`와 같다(17개 종목, 중심·실질·스침 모두).
- `RUN_LLM = False`이면 고른 소식과 LLM 입력만 보여 주고 OpenAI를 부르지 않는다.
- DB 주소는 `.env`의 `GNEWS_DB_URL`(읽기 전용 주소여도 된다), OpenAI 키는 `OPENAI_API_KEY`이며 값은 출력하지 않는다. `.env`는 노트북 폴더에서 위로 올라가며 처음 나오는 것을 쓴다.

In [ ]:
import datetime as dt
import json
import re
import urllib.error
import urllib.request
from pathlib import Path

import sqlalchemy as sa

import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
REASON_PROMPT_PATH = ROOT / 'prompt' / 'company_reason_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 기업별 1문장 정리를 한다
START_KST = '2026-11-19 23:00'   # 시간대 시작(한국 시각, 포함). 브리핑 범위: 전날 23:00
END_KST = '2026-11-20 05:45'     # 시간대 끝(한국 시각, 미포함). 브리핑 범위: 그날 05:45
TIME_COLUMN = 'first_seen_utc'   # 'first_seen_utc'(처음 수집한 시각, 브리핑과 같음) 또는 'published_utc'(발행 시각)
CARD_SLOT = '0730'               # 예상 가격을 읽을 아침 카드(0730: 06:20 계산)
COMPANIES = [            # 정리할 기업(종목코드). news_company_hot_issue와 같은 목록
    '000660',  # SK하이닉스
    '003490',  # 대한항공
    '004370',  # 농심
    '005490',  # POSCO홀딩스
    '009540',  # HD한국조선해양
    '012450',  # 한화에어로스페이스
    '042700',  # 한미반도체
    '068270',  # 셀트리온
    '087010',  # 펩트론
    '090430',  # 아모레퍼시픽
    '105560',  # KB금융
    '105840',  # 우진
    '114090',  # GKL
    '257720',  # 실리콘투
    '259960',  # 크래프톤
    '352820',  # 하이브
    '383800',  # LX홀딩스
]
MENTIONS = ['중심', '실질', '스침']   # entities의 mention 중 이 값인 기사만(중심 / 실질 / 스침)
MAX_ARTICLES_PER_ISSUE = 3
# ==============================

DB_ENV = 'GNEWS_DB_URL'   # 서버 DB 주소(.env 이름). 예상 가격(model.card_stocks)을 읽고, 시간대 안 기사 중 아직 처리 안 된 수를 알린다
ARTICLES_PATH = RESULT_DIR / 'articles.jsonl'       # news_extract가 처리한 기사(DB 확인에 key만 읽는다)
LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'   # 읽기만 한다
IMPORTANCE_PATH = RESULT_DIR / 'importance.csv'     # news_track의 매체 수(읽기만 한다)
COMPANY_REASONS_PATH = RESULT_DIR / 'company_reasons.jsonl'   # 정리할 때마다 기업마다 한 줄
TERMS_PATH = ROOT / 'source' / 'terms' / 'test_queries.csv'   # 검색어 표(종목코드 → 이름, 읽기만)
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'           # LLM 호출마다 토큰 수(다른 노트북과 함께 씀)
KST = dt.timezone(dt.timedelta(hours=9))
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

`news_company_hot_issue.ipynb`와 같은 도구다.
- `env_value`: `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 예상 가격 읽기와 기업마다 소식 고르기

1. **예상 가격**: DB `model.card_stocks`에서 `t` = `END_KST`의 한국 날짜, `slot` = `CARD_SLOT`인 종목별 `h`를 읽는다. `h > 0`이면 `상승`, `h < 0`이면 `하락`, 없거나 0이면 방향 없음.
2. `llm_source.jsonl`에서 `TIME_COLUMN`이 시간대 안인 기사를 **모두** 남긴다(사건으로 묶이지 않은 기사도 기사 하나를 소식 하나로 본다). 순위용 매체 수(`news_outlets`, `story_outlets`)는 **시간대 안 기사로만 다시 센다**(사건으로 묶이지 않은 기사는 1).
3. 기업(종목코드 → 영어 이름)마다 `entities`에 그 기업이 `MENTIONS` 안으로 나온 기사만 남기고, 그 기사의 `company_impact`에서 그 기업의 영향(`impact`)을 가져온다. `스침` 기업은 추출 규칙상 영향이 없다(null).
4. 기사 순위: `news_outlets` 큰 순 → `story_outlets` 큰 순 → 최근 순.
5. 고르기(`pick` 칸):
   - `부호 일치`: 방향이 `상승`이면 영향이 `positive`, `하락`이면 `negative`인 기사 중 1위. `mixed`는 넣지 않는다(2026-11-20 정함).
   - `일치 없음`: 방향은 있는데 맞는 기사가 없으면, 그 기업이 나온 모든 기사 중 1위.
   - `예상 없음`: 예상 가격이 없거나 0이면, 그 기업이 나온 모든 기사 중 1위.
6. 그 기업이 나온 기사가 시간대 안에 없으면 건너뛴다(알린다).
7. LLM 입력: 고른 기사부터, 같은 소식(같은 사건)에서 그 기업이 나온 기사를 `MAX_ARTICLES_PER_ISSUE`건까지.

In [ ]:
def to_utc(text):
    t = pd.Timestamp(text)
    return (t.tz_localize(KST) if t.tzinfo is None else t).tz_convert('UTC')


def db_coverage(start_kst, end_kst):
    """DB(수집 결과)에서 시간대 안 기사(거르기 1단계 통과) 중 news_extract가 아직 처리하지 않은 수를 알린다(2026-11-20 더함).
    요약은 멈추지 않는다. 빠진 기사가 있으면 news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다."""
    url = env_value(DB_ENV)
    if not url:
        print(f'DB 확인 건너뜀: .env에 {DB_ENV}가 없다.')
        return None
    col = {'published_utc': 'published_utc', 'first_seen_utc': 'first_seen_utc'}[TIME_COLUMN]
    engine = sa.create_engine(url)
    try:
        with engine.connect() as con:
            keys = {k for (k,) in con.execute(sa.text(f'select key from items where coalesce("drop", \'\') = \'\' and {col} >= :s and {col} < :e'),
                                              {'s': to_utc(start_kst).to_pydatetime(), 'e': to_utc(end_kst).to_pydatetime()})}
    except Exception as e:
        print('DB 확인 실패(요약은 계속):', type(e).__name__, str(e)[:150])
        return None
    finally:
        engine.dispose()
    done = {r.get('key') for r in read_jsonl(ARTICLES_PATH)}
    todo = len(keys - done)
    print(f'DB 확인: 시간대 안 수집 기사 {len(keys)}건 중 news_extract 처리 {len(keys & done)}건, 아직 처리 안 됨 {todo}건'
          + (' — news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다' if todo else ''))
    return todo


def window_outlets(rows, in_window):
    """순위용 매체 수를 시간대 안 기사로만 다시 센다(2026-11-20 바꿈).
    news_track이 기사마다 남긴 센 출처(source_key)와 소식 번호(news_id)로, 시간대 안 기사만 세어
    news_outlets(소식 매체 수)·story_outlets(사건 매체 수)를 바꾼다. 전체 기간 값은 news_outlets_all·story_outlets_all로 남긴다."""
    rows = rows.copy()
    rows['news_outlets_all'] = pd.to_numeric(rows['news_outlets'], errors='coerce')
    if IMPORTANCE_PATH.exists():
        imp = pd.read_csv(IMPORTANCE_PATH, dtype=str).drop_duplicates('story_id').set_index('story_id')
        rows['story_outlets_all'] = pd.to_numeric(rows['story_id'].map(imp['story_outlets']), errors='coerce')
    else:
        rows['story_outlets_all'] = float('nan')
    if 'source_key' not in rows or rows['source_key'].isna().all():
        print('주의: llm_source.jsonl에 센 출처(source_key)가 없다 — news_track을 한 번 돌려야 한다. 지금은 전체 기간 매체 수로 순위를 매긴다.')
        rows['story_outlets'] = rows['story_outlets_all']
        return rows
    counted = rows[in_window & rows['source_key'].fillna('').astype(str).ne('')]
    news = counted.groupby(['story_id', 'news_id'])['source_key'].nunique()
    story = counted.groupby('story_id')['source_key'].nunique()
    rows['news_outlets'] = [news.get((s, n)) for s, n in zip(rows['story_id'], rows['news_id'])]
    rows['story_outlets'] = rows['story_id'].map(story)
    return rows


def to_int(values):
    return pd.to_numeric(pd.Series(values), errors='coerce').fillna(0).astype(int)


def int_or0(value):
    try:
        return 0 if value is None or pd.isna(value) else int(value)
    except (TypeError, ValueError):
        return 0


def to_kst_text(utc_text):
    """UTC ISO 시각 → 한국 시각 ISO 문자열(예: 2026-11-16T05:42:00+09:00). 못 읽으면 빈 값."""
    t = pd.to_datetime(utc_text, utc=True, errors='coerce')
    return '' if pd.isna(t) else t.tz_convert(KST).isoformat(timespec='seconds')


def text_or(value, default=''):
    return value if isinstance(value, str) and value else default


SUFFIX = re.compile(r'\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|llc|holdings?|group|sa|ag|nv)\b')


def name_key(name):
    """기업 이름 비교용: 소문자, 앞의 the·끝의 Inc./Corp. 등·문장부호·공백을 없앤다."""
    s = (name or '').lower().replace('&', ' and ')
    s = re.sub(r'^the\s+', '', s)
    s = SUFFIX.sub(' ', re.sub(r'[^\w\s]', ' ', s))
    return re.sub(r'\s+', '', s)


def company_names(codes):
    """종목코드 → (한국어 이름, 영어 이름 목록). 검색어 표에서 kind가 '자기 회사'인 검색어(따옴표를 뗀 것)를 영어 이름으로 쓴다."""
    tq = pd.read_csv(TERMS_PATH, dtype=str).fillna('')
    own = tq[tq['kind'] == '자기 회사']
    out, no_name = {}, []
    for code in dict.fromkeys(str(c).strip() for c in codes):   # 같은 코드를 두 번 넣어도 한 번만
        g = own[own['node'] == 'KR:' + code]
        if g.empty:
            no_name.append(code)
            continue
        out[code] = (g['company'].iloc[0], list(dict.fromkeys(q.strip().strip('"') for q in g['query'])))
    if no_name:
        print('주의: 검색어 표에 이름이 없는 종목코드(건너뜀):', no_name)
    return out


def company_entity(row, names):
    """기사(row)의 entities에서 고른 기업(영어 이름 목록 names)과 맞는 기업 entity를 찾는다(MENTIONS 안). 없으면 None."""
    want = {name_key(n) for n in names} - {''}
    for e in row.get('entities') or []:
        if not isinstance(e, dict) or e.get('type') != '기업' or e.get('mention') not in MENTIONS:
            continue
        cand = [e.get('name'), e.get('name_en'), e.get('ticker')] + list(e.get('search_terms') or [])
        if want & {name_key(n) for n in cand if n}:
            return e
    return None


def expected_moves(end_kst, codes):
    """DB model.card_stocks에서 종목별 예상 시가 h(전일 종가 대비 비율)를 읽는다. 못 읽으면 빈 dict(모두 '예상 없음'으로 고른다)."""
    url = env_value(DB_ENV)
    if not url:
        print(f'예상 가격 건너뜀: .env에 {DB_ENV}가 없다.')
        return {}
    day = to_utc(end_kst).tz_convert(KST).strftime('%Y%m%d')
    engine = sa.create_engine(url)
    try:
        with engine.connect() as con:
            got = con.execute(sa.text('select code, h from model.card_stocks where t = :t and slot = :slot and code = any(:codes)'),
                              {'t': day, 'slot': CARD_SLOT, 'codes': list(codes)}).fetchall()
    except Exception as e:
        print('예상 가격 읽기 실패(방향 없이 고름):', type(e).__name__, str(e)[:150])
        return {}
    finally:
        engine.dispose()
    moves = {c: (None if h is None or pd.isna(h) else float(h)) for c, h in got}
    print(f'예상 가격: {day} {CARD_SLOT} 카드에서 {len(moves)}/{len(codes)}종목')
    return moves


def direction_of(h):
    return None if h is None or h == 0 else ('상승' if h > 0 else '하락')


WANT_IMPACT = {'상승': 'positive', '하락': 'negative'}


def pick_company_reasons(start_kst=None, end_kst=None, companies=None):
    """시간대 안 기사(rows)와 기업마다 고른 기사(top, COMPANIES 순서)를 돌려준다."""
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    companies = companies if companies is not None else COMPANIES
    db_coverage(start_kst, end_kst)
    moves = expected_moves(end_kst, companies)
    rows = pd.DataFrame(read_jsonl(LLM_SOURCE_PATH))
    if rows.empty:
        print('llm_source.jsonl이 비어 있다.')
        return rows, rows, {}
    for c in ['entities', 'company_impact', 'story_id', 'news_id', 'source_key', 'news_outlets']:
        if c not in rows:
            rows[c] = None
    sid = rows['story_id'].fillna('').astype(str)
    rows['story_id'] = [s or None for s in sid]
    rows['group'] = [s or 'N:' + k for s, k in zip(sid, rows['key'])]   # 사건으로 묶이지 않은 기사는 기사 하나가 소식 하나
    rows['time'] = pd.to_datetime(rows[TIME_COLUMN], utc=True, format='ISO8601', errors='coerce')
    in_window = rows['time'].ge(to_utc(start_kst)) & rows['time'].lt(to_utc(end_kst))
    rows = window_outlets(rows, in_window)      # 순위용 매체 수: 시간대 안 기사로만
    rows = rows[in_window].copy()
    rows['news_outlets'] = pd.to_numeric(rows['news_outlets'], errors='coerce').fillna(1).astype(int)    # 사건이 아닌 기사는 1
    rows['story_outlets'] = pd.to_numeric(rows['story_outlets'], errors='coerce').fillna(1).astype(int)
    rows = rows.sort_values(['news_outlets', 'story_outlets', 'time'], ascending=[False, False, False]).reset_index(drop=True)

    names = company_names(companies)
    tops, missing, matched = [], [], {}
    for code, (company, en_names) in names.items():
        ents = pd.Series([company_entity(r, en_names) for _, r in rows.iterrows()], index=rows.index, dtype=object)
        cand = rows[ents.notna()].copy()
        if cand.empty:
            missing.append(company)
            continue
        cand['entity'] = ents[cand.index]
        cand['impact'] = [next((x.get('impact') for x in r['company_impact'] or [] if isinstance(x, dict) and x.get('entity_id') == r['entity'].get('id')), None)
                          for _, r in cand.iterrows()]
        h = moves.get(code)
        direction = direction_of(h)
        if direction is None:
            pick, chosen = '예상 없음', cand
        else:
            hit = cand[cand['impact'] == WANT_IMPACT[direction]]
            pick, chosen = ('부호 일치', hit) if len(hit) else ('일치 없음', cand)
        r = chosen.iloc[0].copy()                    # 이미 순위 순으로 정렬돼 있어 맨 앞이 1위
        matched[company] = cand
        r['company_code'], r['company'], r['entity_name'] = code, company, r['entity'].get('name_en') or r['entity'].get('name')
        r['h'], r['direction'], r['pick'], r['company_stories'] = h, direction, pick, cand['group'].nunique()
        tops.append(r)
    top = pd.DataFrame(tops).reset_index(drop=True) if tops else pd.DataFrame()
    print(f'시간대 {start_kst} ~ {end_kst} (KST, {TIME_COLUMN}) | 시간대 안 기사 {len(rows)}건 | '
          f'고른 기업 {len(names)}곳 → 정리할 기업 {len(top)}곳'
          + (f' (시간대 안 기사가 없는 기업: {missing})' if missing else ''))
    return rows, top, matched


def company_input(top, matched):
    """LLM에 넘길 기업 목록(COMPANIES 순서). 고른 기사부터, 같은 소식에서 그 기업이 나온 기사."""
    items = []
    for _, r in top.iterrows():
        cand = matched[r['company']]
        same = pd.concat([cand[cand['key'] == r['key']], cand[(cand['group'] == r['group']) & (cand['key'] != r['key'])]])
        same = same.head(MAX_ARTICLES_PER_ISSUE)
        items.append({'company': r['company'], 'entity_name': r['entity_name'], 'impact': text_or(r['impact']) or None,
                      'news_outlets': int(r['news_outlets']), 'story_outlets': int(r['story_outlets']),
                      'articles': [{'title': a['title'], 'outlet': text_or(a.get('provider'), a['outlet']),
                                    'published_utc': a['published_utc'], 'short_summary': text_or(a.get('short_summary')),
                                    'comparison_summary': text_or(a.get('comparison_summary'))}
                                   for _, a in same.iterrows()]})
    return items

## 4. 기업별 1문장 정리

- 기업마다 고른 소식을 **한 번의 호출**로 OpenAI에 넘겨 `{"lines": [{"company", "line"}, ...]}`(기업마다 한 문장)을 받는다.
- 받은 문장을 `기업: 문장`으로 보여 주고 `company_reasons.jsonl`에 **기업마다 따로 한 줄씩** 남긴다.
  - 공통 칸: 브리핑 날짜 `date`(시간대 끝의 한국 날짜), 생성 시각 `created_kst`(한국), 시간대 `start_kst`·`end_kst`, `time_column`, `card_slot`, `model`
  - 기업 칸: `company_code`, `company`, `entity_name`, `h`(예상 시가, 전일 종가 대비 비율), `direction`(상승/하락/null), `pick`(부호 일치/일치 없음/예상 없음), `impact`(고른 기사의 그 기업 영향), `story_id`, `key`(고른 기사), `published_kst`, `news_outlets`, `story_outlets`, `company_stories`(그 기업이 나온 소식 수), `news_outlets_all`, `story_outlets_all`, `line`(1문장)
- 받은 기업 이름이 입력과 다르거나 빠지면 알리고, 그 기업은 저장하지 않는다. 정리할 기업이 없으면 OpenAI를 부르지 않는다.

In [ ]:
REASON_PROMPT = REASON_PROMPT_PATH.read_text(encoding='utf-8')


def company_reasons(start_kst=None, end_kst=None, companies=None):
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    rows, top, matched = pick_company_reasons(start_kst, end_kst, companies)
    if top.empty:
        print('정리할 기업이 없다.')
        return {}
    display(top[['company_code', 'company', 'h', 'direction', 'pick', 'impact', 'entity_name', 'story_id', 'news_outlets', 'story_outlets',
                 'company_stories', 'published_utc', 'title']])
    items = company_input(top, matched)
    if not RUN_LLM:
        print('RUN_LLM = False라 OpenAI를 부르지 않는다. LLM에 넘길 입력:')
        print(json.dumps({'companies': items}, ensure_ascii=False, indent=1))
        return {}
    text, usage = ask_openai(REASON_PROMPT, 'companies json:\n' + json.dumps({'companies': items}, ensure_ascii=False), {'type': 'json_object'})
    log_usage('company_reason', f'{start_kst}~{end_kst}', usage)
    got = {}
    for x in json.loads(text).get('lines', []):
        if isinstance(x, dict) and x.get('company'):
            got[str(x['company']).strip()] = str(x.get('line', '')).strip().lstrip('-•· ').strip()
    wrong = sorted(set(got) - set(top['company'])) + [c for c in top['company'] if not got.get(c)]
    if wrong:
        print('주의: 입력과 맞지 않거나 빠진 기업(저장하지 않음):', wrong)
    now = dt.datetime.now(dt.timezone.utc)
    common = {'date': to_utc(end_kst).tz_convert(KST).strftime('%Y-%m-%d'),   # 브리핑 날짜(시간대 끝의 한국 날짜)
              'created_kst': now.astimezone(KST).strftime('%Y-%m-%d %H:%M:%S'),   # 생성 시각(한국)
              'start_kst': start_kst, 'end_kst': end_kst, 'time_column': TIME_COLUMN, 'card_slot': CARD_SLOT, 'model': LLM_MODEL}
    out = [{**common, 'company_code': r['company_code'], 'company': r['company'], 'entity_name': r['entity_name'],
            'h': None if r['h'] is None or pd.isna(r['h']) else float(r['h']), 'direction': r['direction'], 'pick': r['pick'],
            'impact': text_or(r['impact']) or None, 'story_id': text_or(r['story_id']) or None, 'key': r['key'],
            'published_kst': to_kst_text(r['published_utc']), 'news_outlets': int(r['news_outlets']),
            'story_outlets': int(r['story_outlets']), 'company_stories': int(r['company_stories']),
            'news_outlets_all': int_or0(r['news_outlets_all']), 'story_outlets_all': int_or0(r['story_outlets_all']),
            'line': got[r['company']]}
           for _, r in top.iterrows() if got.get(r['company'])]
    append_jsonl(COMPANY_REASONS_PATH, out)              # 기업마다 따로 한 줄(공통 칸을 줄마다 붙임)
    for r in out:
        print(f"{r['company']} ({r['direction'] or '방향 없음'}, {r['pick']}): {r['line']}")
    return {r['company']: r['line'] for r in out}

## 5. 실행

In [ ]:
company_reason_lines = company_reasons()